---
## § 0. Smart Checkpoint & Resume 

In [1]:
"""
SMART CHECKPOINT & RESUME SYSTEM
=================================
Features:
  - Multi-stage checkpointing (data prep, training, auditing)
  - Automatic resume after connection loss or interruption
  - Intelligent decision: resume vs fresh start
  - Detects model architecture changes, config changes, data changes
  - Quality assessment: abandons bad runs automatically
  - Informative decision messages at every stage
"""

import os, sys, json, hashlib, warnings, gc, time, random, math, shutil
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict
from enum import Enum

import numpy as np
import torch
import torch.nn as nn

warnings.filterwarnings('ignore')


class RunDecision(Enum):
    RESUME_TRAINING = "resume_training"
    RESUME_AUDIT = "resume_audit"
    RESUME_PLOTTING = "resume_plotting"
    START_FRESH = "start_fresh"
    COMPLETE = "complete"


class CheckpointManager:
    """
    Manages all checkpointing logic for the experiment.
    Handles save/load/decision-making for resume vs fresh start.
    """

    def __init__(self, exp_dir: Path, experiment_name: str = "paper_accurate"):
        self.exp_dir = exp_dir
        self.ckpt_dir = exp_dir / "checkpoints"
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)

        self.experiment_name = experiment_name
        self.metadata_file = self.ckpt_dir / "metadata.json"
        self.decision_log_file = self.ckpt_dir / "decision_log.json"
        self.config_hash_file = self.ckpt_dir / "config_hash.txt"
        self.model_hash_file = self.ckpt_dir / "model_hash.txt"

        self.decision_log = []
        self._load_decision_log()

    # ──────────────────────────────────────────────────────────────────
    # HASHING: Detect changes in config / model / data
    # ──────────────────────────────────────────────────────────────────

    def compute_config_hash(self, cfg: dict) -> str:
        """Hash the experiment config to detect changes."""
        relevant_keys = [
            'epochs', 'lr', 'batch_size', 'weight_decay', 'val_frac', 'seed',
            'dro_mode', 'dro_alpha', 'dro_eta', 'dro_lambda_init', 'dro_warmup_epochs',
            'dp_noise_multiplier', 'dp_max_grad_norm', 'dp_delta', 'dp_phys_batch',
            'audit_k', 'audit_gamma', 'audit_seed'
        ]
        hash_input = {k: str(cfg.get(k, '')) for k in relevant_keys}
        raw = json.dumps(hash_input, sort_keys=True)
        return hashlib.sha256(raw.encode()).hexdigest()[:16]

    def compute_model_hash(self, model: nn.Module) -> str:
        """Hash model architecture (layer names + shapes) to detect changes."""
        arch_str = ""
        for name, param in model.named_parameters():
            arch_str += f"{name}:{list(param.shape)};"
        return hashlib.sha256(arch_str.encode()).hexdigest()[:16]

    def save_hashes(self, cfg: dict, model: nn.Module):
        """Save config and model hashes for future comparison."""
        cfg_hash = self.compute_config_hash(cfg)
        model_hash = self.compute_model_hash(model)

        self.config_hash_file.write_text(cfg_hash)
        self.model_hash_file.write_text(model_hash)
        return cfg_hash, model_hash

    def load_previous_hashes(self) -> Tuple[Optional[str], Optional[str]]:
        """Load previously saved hashes. Returns (None, None) if not found."""
        cfg_hash = None
        model_hash = None
        if self.config_hash_file.exists():
            cfg_hash = self.config_hash_file.read_text().strip()
        if self.model_hash_file.exists():
            model_hash = self.model_hash_file.read_text().strip()
        return cfg_hash, model_hash

    # ──────────────────────────────────────────────────────────────────
    # METADATA: Track experiment progress
    # ──────────────────────────────────────────────────────────────────

    def save_metadata(self, metadata: dict):
        """Save experiment metadata."""
        metadata['last_updated'] = datetime.now().isoformat()
        with open(self.metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2)

    def load_metadata(self) -> Optional[dict]:
        """Load experiment metadata. Returns None if not found."""
        if self.metadata_file.exists():
            with open(self.metadata_file, 'r') as f:
                return json.load(f)
        return None

    def _load_decision_log(self):
        """Load previous decision log."""
        if self.decision_log_file.exists():
            with open(self.decision_log_file, 'r') as f:
                self.decision_log = json.load(f)

    def log_decision(self, decision: RunDecision, reason: str, details: dict = None):
        """Log a resume/fresh decision with timestamp and reason."""
        entry = {
            'timestamp': datetime.now().isoformat(),
            'decision': decision.value,
            'reason': reason,
            'details': details or {}
        }
        self.decision_log.append(entry)
        with open(self.decision_log_file, 'w') as f:
            json.dump(self.decision_log, f, indent=2)

    # ──────────────────────────────────────────────────────────────────
    # CHECKPOINT SAVE/LOAD: Model + Optimizer + State
    # ──────────────────────────────────────────────────────────────────

    def save_training_checkpoint(
        self,
        phase: str,           # 'erm' or 'alg1'
        epoch: int,
        model: nn.Module,
        optimizer: torch.optim.Optimizer,
        history: dict,
        dro_state: dict = None,
        privacy_engine_steps: int = 0,
        extra: dict = None
    ):
        """Save a training checkpoint at a specific epoch."""
        phase_dir = self.ckpt_dir / phase
        phase_dir.mkdir(parents=True, exist_ok=True)

        base_model = model._module if hasattr(model, '_module') else model
        state = {
            'epoch': epoch,
            'phase': phase,
            'model_state_dict': base_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history,
            'dro_state': dro_state or {},
            'privacy_engine_steps': privacy_engine_steps,
            'extra': extra or {},
            'timestamp': datetime.now().isoformat()
        }
        ckpt_path = phase_dir / f"epoch_{epoch:03d}.pt"
        torch.save(state, ckpt_path)

        # Also save as 'latest' for quick resume
        latest_path = phase_dir / "latest.pt"
        torch.save(state, latest_path)

        return ckpt_path

    def load_training_checkpoint(self, phase: str, epoch: Optional[int] = None) -> Optional[dict]:
        """Load a training checkpoint. If epoch is None, loads latest."""
        phase_dir = self.ckpt_dir / phase

        if epoch is not None:
            ckpt_path = phase_dir / f"epoch_{epoch:03d}.pt"
        else:
            ckpt_path = phase_dir / "latest.pt"

        if ckpt_path.exists():
            return torch.load(ckpt_path, map_location='cpu')
        return None

    def save_data_split(self, train_idx, val_idx, canary_in_pool, canary_out_pool, membership):
        """Save data split indices so resume uses same split."""
        split_path = self.ckpt_dir / "data_split.pt"
        torch.save({
            'train_idx': train_idx,
            'val_idx': val_idx,
            'canary_in_pool': canary_in_pool,
            'canary_out_pool': canary_out_pool,
            'membership': membership,
        }, split_path)

    def load_data_split(self) -> Optional[dict]:
        """Load saved data split."""
        split_path = self.ckpt_dir / "data_split.pt"
        if split_path.exists():
            return torch.load(split_path, map_location='cpu')
        return None

    def save_audit_results(self, audit_data: dict):
        """Save audit results."""
        audit_path = self.ckpt_dir / "audit_results.json"
        with open(audit_path, 'w') as f:
            json.dump(audit_data, f, indent=2)

    def load_audit_results(self) -> Optional[dict]:
        """Load saved audit results."""
        audit_path = self.ckpt_dir / "audit_results.json"
        if audit_path.exists():
            with open(audit_path, 'r') as f:
                return json.load(f)
        return None

    def save_final_results(self, results: dict):
        """Save final consolidated results."""
        final_path = self.ckpt_dir / "final_results.json"
        with open(final_path, 'w') as f:
            json.dump(results, f, indent=2)

    def load_final_results(self) -> Optional[dict]:
        """Load final results if they exist."""
        final_path = self.ckpt_dir / "final_results.json"
        if final_path.exists():
            with open(final_path, 'r') as f:
                return json.load(f)
        return None

    # ──────────────────────────────────────────────────────────────────
    # QUALITY ASSESSMENT: Decide if results are worth keeping
    # ──────────────────────────────────────────────────────────────────

    def assess_training_quality(self, history: dict, phase: str) -> Tuple[bool, str]:
        """
        Assess if training results so far are worth keeping.
        Returns (is_worth_keeping, reason_string).
        """
        issues = []

        # Check 1: Loss is NaN or Inf
        if 'train_loss' in history and len(history['train_loss']) > 0:
            losses = history['train_loss']
            if any(math.isnan(l) or math.isinf(l) for l in losses):
                issues.append("Loss contains NaN or Inf values")

        # Check 2: MAE is reasonable (not exploding)
        if 'overall_mae' in history and len(history['overall_mae']) > 0:
            latest_mae = history['overall_mae'][-1]
            if latest_mae > 100:  # Unreasonable for age regression
                issues.append(f"MAE is unreasonably high: {latest_mae:.1f}")
            if latest_mae < 0.1:  # Suspiciously low
                issues.append(f"MAE is suspiciously low: {latest_mae:.4f} (possible data leak)")

        # Check 3: Epsilon is within expected range
        if 'epsilon' in history and len(history['epsilon']) > 0:
            latest_eps = history['epsilon'][-1]
            if latest_eps > 100:
                issues.append(f"Epsilon is extremely high: {latest_eps:.1f}")

        # Check 4: Training is actually converging (loss decreasing over epochs)
        if 'train_loss' in history and len(history['train_loss']) >= 5:
            first_half = np.mean(history['train_loss'][:len(history['train_loss'])//2])
            second_half = np.mean(history['train_loss'][len(history['train_loss'])//2:])
            if second_half > first_half * 1.5:  # Loss increased significantly
                issues.append(f"Loss is diverging: first_half={first_half:.2f}, second_half={second_half:.2f}")

        # Check 5: Gender gap is not exploding
        if 'gender_gap' in history and len(history['gender_gap']) > 0:
            if history['gender_gap'][-1] > 50:
                issues.append(f"Gender gap is unreasonably high: {history['gender_gap'][-1]:.1f}")

        # Check 6: Lambda_t for DRO phase
        if phase == 'alg1' and 'lambda_t' in history and len(history['lambda_t']) > 0:
            # This is informational, not a failure condition
            if all(l == 0.0 for l in history['lambda_t']):
                pass  # Informational only, not a failure

        if issues:
            return False, "; ".join(issues)
        return True, "All quality checks passed"

    # ──────────────────────────────────────────────────────────────────
    # MAIN DECISION ENGINE: Resume or Fresh?
    # ──────────────────────────────────────────────────────────────────

    def decide_action(self, cfg: dict, model: nn.Module) -> Tuple[RunDecision, str, dict]:
        """
        Main decision engine. Examines all saved state and decides
        whether to resume or start fresh.

        Returns: (decision, human_readable_reason, context_dict)
        """
        metadata = self.load_metadata()
        prev_cfg_hash, prev_model_hash = self.load_previous_hashes()
        curr_cfg_hash = self.compute_config_hash(cfg)
        curr_model_hash = self.compute_model_hash(model)

        context = {
            'curr_cfg_hash': curr_cfg_hash,
            'curr_model_hash': curr_model_hash,
            'prev_cfg_hash': prev_cfg_hash,
            'prev_model_hash': prev_model_hash,
        }

        # ── Case 1: No previous run exists ──
        if metadata is None:
            reason = "No previous run found. Starting fresh."
            self.log_decision(RunDecision.START_FRESH, reason)
            return RunDecision.START_FRESH, reason, context

        # ── Case 2: Model architecture changed ──
        if prev_model_hash is not None and prev_model_hash != curr_model_hash:
            reason = (
                "STARTING FRESH: Model architecture has changed since last run. "
                f"Previous model hash: {prev_model_hash}, Current: {curr_model_hash}. "
                "Cannot resume with a different model structure."
            )
            self.log_decision(RunDecision.START_FRESH, reason, context)
            return RunDecision.START_FRESH, reason, context

        # ── Case 3: Config changed ──
        if prev_cfg_hash is not None and prev_cfg_hash != curr_cfg_hash:
            reason = (
                "STARTING FRESH: Experiment configuration has changed since last run. "
                f"Previous config hash: {prev_cfg_hash}, Current: {curr_cfg_hash}. "
                "Hyperparameters were modified, so previous results are not comparable."
            )
            self.log_decision(RunDecision.START_FRESH, reason, context)
            return RunDecision.START_FRESH, reason, context

        # ── Case 4: Previous run completed everything ──
        if metadata.get('status') == 'complete':
            final_results = self.load_final_results()
            if final_results is not None:
                reason = (
                    "EXPERIMENT ALREADY COMPLETE: All training, auditing, and plotting "
                    "were completed in a previous run. Loading saved results. "
                    "Delete the checkpoint folder to force a fresh run."
                )
                self.log_decision(RunDecision.COMPLETE, reason, context)
                return RunDecision.COMPLETE, reason, context

        # ── Case 5: Check what stage was completed ──
        completed_phases = metadata.get('completed_phases', [])
        current_stage = metadata.get('current_stage', 'none')

        # Check quality of saved training results
        for phase in ['erm', 'alg1']:
            ckpt = self.load_training_checkpoint(phase)
            if ckpt is not None and phase in completed_phases:
                is_good, quality_reason = self.assess_training_quality(ckpt['history'], phase)
                if not is_good:
                    reason = (
                        f"STARTING FRESH: Previous {phase.upper()} training results failed "
                        f"quality assessment. Reason: {quality_reason}. "
                        "The notebook decided these results are not worth keeping."
                    )
                    self.log_decision(RunDecision.START_FRESH, reason, context)
                    return RunDecision.START_FRESH, reason, context

        # ── Case 6: Determine resume point ──
        if 'audit' in completed_phases and 'plotting' not in completed_phases:
            reason = (
                "RESUMING FROM PLOTTING PHASE: Training and auditing were completed "
                "in a previous run. Resuming from the plotting/visualization stage. "
                "All model weights and audit results are preserved."
            )
            self.log_decision(RunDecision.RESUME_PLOTTING, reason, context)
            return RunDecision.RESUME_PLOTTING, reason, context

        if 'alg1_training' in completed_phases and 'audit' not in completed_phases:
            reason = (
                "RESUMING FROM AUDIT PHASE: Both ERM and ALG1 training were completed "
                "and passed quality checks in a previous run. Resuming from the "
                "privacy audit stage. Model weights are preserved."
            )
            self.log_decision(RunDecision.RESUME_AUDIT, reason, context)
            return RunDecision.RESUME_AUDIT, reason, context

        if 'erm_training' in completed_phases and 'alg1_training' not in completed_phases:
            # Check ERM quality
            erm_ckpt = self.load_training_checkpoint('erm')
            if erm_ckpt is not None:
                is_good, quality_reason = self.assess_training_quality(erm_ckpt['history'], 'erm')
                if is_good:
                    reason = (
                        "RESUMING FROM ALG1 TRAINING: ERM training was completed and passed "
                        "quality checks in a previous run. A previous code block accepted the "
                        "ERM results. Now resuming from ALG1 (DRO) training. "
                        f"ERM completed {erm_ckpt['epoch']} epochs with "
                        f"final MAE={erm_ckpt['history']['overall_mae'][-1]:.2f}."
                    )
                    self.log_decision(RunDecision.RESUME_TRAINING, reason, context)
                    return RunDecision.RESUME_TRAINING, reason, context
                else:
                    reason = (
                        f"STARTING FRESH: ERM training was completed but failed quality "
                        f"assessment: {quality_reason}. Starting over."
                    )
                    self.log_decision(RunDecision.START_FRESH, reason, context)
                    return RunDecision.START_FRESH, reason, context

        # ── Case 7: Partial training (interrupted mid-epoch) ──
        if current_stage in ['erm_training', 'alg1_training']:
            phase = 'erm' if 'erm' in current_stage else 'alg1'
            ckpt = self.load_training_checkpoint(phase)

            if ckpt is not None:
                is_good, quality_reason = self.assess_training_quality(ckpt['history'], phase)
                if is_good:
                    reason = (
                        f"RESUMING {phase.upper()} TRAINING from epoch {ckpt['epoch']}: "
                        f"Training was interrupted (possibly due to connection loss or "
                        f"manual stop). The previous code block accepted results up to "
                        f"epoch {ckpt['epoch']}. Quality check passed: {quality_reason}. "
                        f"Resuming from epoch {ckpt['epoch'] + 1}."
                    )
                    self.log_decision(RunDecision.RESUME_TRAINING, reason, context)
                    return RunDecision.RESUME_TRAINING, reason, context
                else:
                    reason = (
                        f"STARTING FRESH: {phase.upper()} training was interrupted at epoch "
                        f"{ckpt['epoch']}, but the results failed quality assessment: "
                        f"{quality_reason}. The notebook decided these partial results are "
                        f"not worth keeping. Starting over."
                    )
                    self.log_decision(RunDecision.START_FRESH, reason, context)
                    return RunDecision.START_FRESH, reason, context

        # ── Default: Start fresh ──
        reason = "STARTING FRESH: Could not determine a valid resume point. Starting over."
        self.log_decision(RunDecision.START_FRESH, reason, context)
        return RunDecision.START_FRESH, reason, context

    # ──────────────────────────────────────────────────────────────────
    # CLEANUP: Archive old checkpoints
    # ──────────────────────────────────────────────────────────────────

    def archive_previous_run(self):
        """Archive the previous checkpoint folder before starting fresh."""
        if self.ckpt_dir.exists() and any(self.ckpt_dir.iterdir()):
            archive_name = f"checkpoints_archived_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            archive_path = self.exp_dir / archive_name
            shutil.move(str(self.ckpt_dir), str(archive_path))
            self.ckpt_dir.mkdir(parents=True, exist_ok=True)
            return archive_path
        return None

    def print_decision(self, decision: RunDecision, reason: str):
        """Print a formatted decision message."""
        print("\n" + "═" * 78)
        if decision == RunDecision.START_FRESH:
            print("  🔄 DECISION: STARTING FRESH RUN")
        elif decision == RunDecision.RESUME_TRAINING:
            print("  ▶️  DECISION: RESUMING TRAINING")
        elif decision == RunDecision.RESUME_AUDIT:
            print("  ▶️  DECISION: RESUMING FROM AUDIT PHASE")
        elif decision == RunDecision.RESUME_PLOTTING:
            print("  ▶️  DECISION: RESUMING FROM PLOTTING PHASE")
        elif decision == RunDecision.COMPLETE:
            print("  ✅ DECISION: EXPERIMENT ALREADY COMPLETE")
        print("═" * 78)
        print(f"  Reason: {reason}")
        print("═" * 78 + "\n")


# ──────────────────────────────────────────────────────────────────────
# INSTANTIATE THE CHECKPOINT MANAGER
# ──────────────────────────────────────────────────────────────────────
EXP_DIR = Path('project/experiments/paper_accurate_final')
CKPT = CheckpointManager(EXP_DIR, experiment_name="paper_accurate")

print("✅ Smart Checkpoint System initialized.")
print(f"   Checkpoint directory: {CKPT.ckpt_dir}")
print(f"   Decision log: {CKPT.decision_log_file}")

✅ Smart Checkpoint System initialized.
   Checkpoint directory: project/experiments/paper_accurate_final/checkpoints
   Decision log: project/experiments/paper_accurate_final/checkpoints/decision_log.json


## § 1. Imports, Device & Configuration

In [2]:
"""
Paper-Accurate Hashimoto et al. (2018) Algorithm 1 + DP-SGD + Steinke Audit
===========================================================================
With Smart Checkpoint & Resume System
"""

import os, sys, json, warnings, gc, time, random, math
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset, ConcatDataset, WeightedRandomSampler
from torchvision import transforms

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from opacus.utils.batch_memory_manager import BatchMemoryManager

warnings.filterwarnings('ignore')

# Fix Opacus BatchMemoryManager
def _patched_batch_memory_iter(self):
    for batch in self.data_loader:
        for i in range(0, len(batch[0]), self.max_physical_batch_size):
            yield tuple(x[i : i + self.max_physical_batch_size] for x in batch)
BatchMemoryManager.__iter__ = _patched_batch_memory_iter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} - {props.total_memory / 1e9:.1f} GB')

RACE_MAP = {0: 'White', 1: 'Black', 2: 'Asian', 3: 'Indian', 4: 'Others'}
RACE_NAMES = ['white', 'black', 'asian', 'indian', 'others']

CFG = {
    'image_dir':    Path('project/data/UTKFace'),
    'exp_dir':      EXP_DIR,
    'epochs':       25,
    'lr':           1e-4,
    'batch_size':   64,
    'weight_decay': 1e-4,
    'val_frac':     0.15,
    'seed':         2024,

    # Hashimoto Algorithm 1 Parameters
    'dro_mode':            'alg1_chisq',
    'dro_alpha':           1.0,
    'dro_eta':             0.01,
    'dro_warmup_epochs':   1,

    # DP-SGD
    'dp_noise_multiplier': 1.1,
    'dp_max_grad_norm':    1.0,
    'dp_delta':            1e-5,
    'dp_phys_batch':       32,

    # Privacy Auditing
    'audit_k':             500,
    'audit_gamma':         0.05,
    'audit_seed':          42,
    'wb_score_epochs':     5,
    'wb_sample_batches':   10,

    # Checkpoint frequency
    'checkpoint_every_n_epochs': 5,  # Save checkpoint every N epochs
}

EXP_DIR.mkdir(parents=True, exist_ok=True)
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])

print("✅ Configuration loaded.")

Device: cuda
GPU: NVIDIA L40S - 47.7 GB
✅ Configuration loaded.


## § 2. Dataset & Model Definition

In [3]:
# ══════════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════════
class UTKFaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir, self.transform, self.samples = Path(root_dir), transform, []
        for p in sorted(self.root_dir.rglob('*.jpg')):
            try:
                parts = p.stem.replace('.chip', '').split('_')
                if len(parts) >= 4:
                    age, gender, race = int(parts[0]), int(parts[1]), int(parts[2])
                    if 1 <= age <= 116 and gender in (0, 1) and race in RACE_MAP:
                        self.samples.append({'path': str(p), 'age': age, 'gender': gender, 'race': race})
            except:
                pass

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(s['age'], dtype=torch.float32), \
               torch.tensor(s['gender'], dtype=torch.long), \
               torch.tensor(s['race'], dtype=torch.long)

    def get_pil_image(self, idx):
        return Image.open(self.samples[idx]['path']).convert('RGB')


train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

canary_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('Loading UTKFace dataset...')
full_ds = UTKFaceDataset(CFG['image_dir'], transform=None)
N = len(full_ds)
print(f'  Total samples: {N}')

# ══════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════
class GNBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, 3, stride=stride, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(min(32, planes), planes)
        self.relu = nn.ReLU(inplace=False)
        self.conv2 = nn.Conv2d(planes, planes, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(min(32, planes), planes)
        self.downsample = downsample

    def forward(self, x):
        out = self.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        if self.downsample:
            x = self.downsample(x)
        return self.relu(out + x)


class OpacusResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False)
        self.gn1 = nn.GroupNorm(min(32, 64), 64)
        self.relu = nn.ReLU(inplace=False)
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, 1)
        self.layer2 = self._make_layer(128, 2, 2)
        self.layer3 = self._make_layer(256, 2, 2)
        self.layer4 = self._make_layer(512, 2, 2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, 1)

    def _make_layer(self, planes, blocks, stride):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, 1, stride=stride, bias=False),
                nn.GroupNorm(min(32, planes), planes)
            )
        layers = [GNBlock(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(GNBlock(planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.gn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(torch.flatten(self.avgpool(x), 1)).squeeze(1)


def build_model():
    m = OpacusResNet18()
    if ModuleValidator.validate(m, strict=False):
        m = ModuleValidator.fix(m)
    return m.to(device)

print("✅ Model architecture defined.")

Loading UTKFace dataset...
  Total samples: 33483
✅ Model architecture defined.


## § 3. Smart Data Splitting (with checkpoint save/load)

In [4]:
# ══════════════════════════════════════════════════════════════════
# SMART DATA SPLITTING
# ══════════════════════════════════════════════════════════════════
k = CFG['audit_k']

# Check if we have a saved data split from a previous run
saved_split = CKPT.load_data_split()

if saved_split is not None:
    print("  📂 Found saved data split from previous run. Reusing same split for consistency.")
    train_idx = saved_split['train_idx']
    val_idx = saved_split['val_idx']
    canary_in_pool = saved_split['canary_in_pool']
    canary_out_pool = saved_split['canary_out_pool']
    membership = saved_split['membership']
else:
    print("  🆕 Creating new data split.")
    audit_rng = np.random.default_rng(CFG['audit_seed'])
    all_indices = list(range(N))
    audit_rng.shuffle(all_indices)
    canary_in_pool = all_indices[:k]
    canary_out_pool = all_indices[k:2*k]
    remaining_indices = all_indices[2*k:]
    n_val = int(len(remaining_indices) * CFG['val_frac'])

    gen = torch.Generator().manual_seed(CFG['seed'])
    train_idx, val_idx = random_split(
        remaining_indices,
        [len(remaining_indices) - n_val, n_val],
        generator=gen
    )
    train_idx = list(train_idx)
    val_idx = list(val_idx)
    membership = torch.tensor(
        audit_rng.integers(0, 2, size=k).astype(bool),
        dtype=torch.bool
    )

    # Save the split for future resume
    CKPT.save_data_split(train_idx, val_idx, canary_in_pool, canary_out_pool, membership)
    print(f"  💾 Data split saved to checkpoint.")

print(f"  Train: {len(train_idx)}, Val: {len(val_idx)}, Canary IN: {k}, Canary OUT: {k}")
print(f"  Membership balance: {membership.sum().item()} IN / {(~membership).sum().item()} OUT")


class TransformSubset(Dataset):
    def __init__(self, base, indices, transform):
        self.base, self.indices, self.tf = base, indices, transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, age, g, r = self.base[self.indices[i]]
        return self.tf(img), age, g, r


train_ds = TransformSubset(full_ds, train_idx, train_tf)
val_ds = TransformSubset(full_ds, val_idx, val_tf)

print("✅ Data split complete.")

  🆕 Creating new data split.
  💾 Data split saved to checkpoint.
  Train: 27611, Val: 4872, Canary IN: 500, Canary OUT: 500
  Membership balance: 243 IN / 257 OUT
✅ Data split complete.


## § 4. Canary Dataset

In [5]:
# ══════════════════════════════════════════════════════════════════
# CANARY DATASET
# ══════════════════════════════════════════════════════════════════
class CanaryDataset:
    def __init__(self, base_ds, in_idx, out_idx, membership, transform, rng):
        self.base_ds = base_ds
        self.membership = membership
        self.k = len(membership)
        self.image_indices = [in_idx[i] if m else out_idx[i] for i, m in enumerate(membership)]
        held_out_ages = [base_ds.samples[i]['age'] for i in val_idx]
        rng.shuffle(held_out_ages)
        self.shuffled_ages = torch.tensor(held_out_ages[:self.k], dtype=torch.float32)
        print(f'  Pre-computing {self.k} canary images...')
        self.canary_images = [transform(base_ds.get_pil_image(idx)) for idx in self.image_indices]

    def get_in_dataset(self):
        in_idxs = [i for i, m in enumerate(self.membership) if m]
        return _InCanarySubset(self, in_idxs), in_idxs

    def get_all_canary_batch(self):
        return torch.stack(self.canary_images).to(device), self.shuffled_ages.to(device)


class _InCanarySubset(Dataset):
    def __init__(self, cd, in_idxs):
        self.cd = cd
        self.in_idxs = in_idxs
    def __len__(self):
        return len(self.in_idxs)
    def __getitem__(self, i):
        img = self.cd.canary_images[self.in_idxs[i]].clone()
        age = self.cd.shuffled_ages[self.in_idxs[i]]
        return img, age, torch.tensor(-1, dtype=torch.long), torch.tensor(-1, dtype=torch.long)


audit_rng = np.random.default_rng(CFG['audit_seed'])
canary_ds = CanaryDataset(full_ds, canary_in_pool, canary_out_pool, membership, canary_tf, audit_rng)
in_canary_ds, in_canary_global_idxs = canary_ds.get_in_dataset()
augmented_ds = ConcatDataset([train_ds, in_canary_ds])

print("✅ Canary dataset ready.")

  Pre-computing 500 canary images...
✅ Canary dataset ready.


## § 5. Evaluation & Audit Utilities

In [6]:
# ══════════════════════════════════════════════════════════════════
# EVALUATION & AUDIT UTILITIES
# ══════════════════════════════════════════════════════════════════
def evaluate_fairness(model, loader):
    model.eval()
    gender_loss = {0: 0.0, 1: 0.0}
    gender_count = {0: 0, 1: 0}
    race_loss = {r: 0.0 for r in RACE_MAP}
    race_count = {r: 0 for r in RACE_MAP}
    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for imgs, ages, genders, races in loader:
            imgs, ages = imgs.to(device), ages.to(device)
            abs_err = (model(imgs) - ages).abs().cpu()
            valid = (genders >= 0)
            total_loss += abs_err[valid].sum().item()
            total_count += valid.sum().item()
            for g in (0, 1):
                mask = (genders == g) & valid
                gender_loss[g] += abs_err[mask].sum().item()
                gender_count[g] += mask.sum().item()
            for r in RACE_MAP:
                mask = (races == r) & valid
                race_loss[r] += abs_err[mask].sum().item()
                race_count[r] += mask.sum().item()

    res = {
        'overall_mae': total_loss / max(total_count, 1),
        'male_mae': gender_loss[0] / max(gender_count[0], 1),
        'female_mae': gender_loss[1] / max(gender_count[1], 1),
        'gender_gap': abs(gender_loss[0] / max(gender_count[0], 1) - gender_loss[1] / max(gender_count[1], 1))
    }
    for race_name in RACE_NAMES:
        race_id = [k for k, v in RACE_MAP.items() if v.lower() == race_name][0]
        res[f'{race_name}_mae'] = race_loss[race_id] / max(race_count[race_id], 1)
    race_maes = [res[f'{r}_mae'] for r in RACE_NAMES]
    res['race_gap'] = max(race_maes) - min(race_maes)
    return res


def audit_epsilon(scores, membership, gamma=0.05):
    m = membership.numpy().astype(int)
    k = len(scores)
    thresholds = np.linspace(scores.min() - 1e-9, scores.max() + 1e-9, 1000)
    best_V = max(((scores > tau).astype(int) == m).sum() for tau in thresholds)
    p_null = stats.binom.sf(best_V - 1, k, 0.5)

    def p_val(eps):
        p_succ = math.exp(eps) / (1 + math.exp(eps))
        return stats.binom.sf(best_V - 1, k, p_succ)

    lo, hi = 0.0, 20.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if p_val(mid) > gamma:
            lo = mid
        else:
            hi = mid

    in_s = scores[m == 1]
    out_s = scores[m == 0]
    pooled_std = np.sqrt((in_s.var() + out_s.var()) / 2)

    try:
        from sklearn.metrics import roc_auc_score
        auc = roc_auc_score(m, scores)
    except:
        auc = 0.5

    return {
        'eps_lb': lo if p_val(lo) <= gamma else 0.0,
        'V': int(best_V),
        'k': k,
        'threshold': float(thresholds[0]),
        'p_value': float(p_null),
        'attack_acc': best_V / k,
        'delta': float(in_s.mean() - out_s.mean()),
        'cohens_d': float((in_s.mean() - out_s.mean()) / (pooled_std + 1e-8)),
        'auc': float(auc)
    }

print("✅ Evaluation & audit utilities ready.")

✅ Evaluation & audit utilities ready.


## § 6. Smart Training Function (with checkpoint save/load & emergency save)

In [7]:
# ══════════════════════════════════════════════════════════════════
# SMART TRAINING FUNCTION WITH CHECKPOINTING
# ══════════════════════════════════════════════════════════════════
def train_model_with_checkpoints(
    cfg, train_ds, val_loader, canary_ds, membership,
    phase='erm', experiment_name='exp', resume_from_epoch=0
):
    """
    Training function with smart checkpointing.
    Saves checkpoints every N epochs and on interruption.
    Supports resume from a specific epoch.
    """
    n_train = len(train_ds)
    k = len(membership)
    alpha = cfg['dro_alpha']
    eta = cfg['dro_eta']
    is_alg1 = (cfg['dro_mode'] == 'alg1_chisq')
    checkpoint_every = cfg.get('checkpoint_every_n_epochs', 5)

    # Initialize Algorithm 1 variables
    q_t = np.ones(n_train) / n_train
    lambda_t = 0.0

    # Data loader
    def get_uniform_loader(dataset):
        return DataLoader(dataset, batch_size=cfg['batch_size'], shuffle=True,
                         num_workers=4, pin_memory=True)
    train_loader = get_uniform_loader(augmented_ds)

    # Build model and optimizer
    model = build_model()
    optimizer = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    criterion = nn.L1Loss(reduction='mean')

    # ── RESUME: Load checkpoint if resuming ──
    start_epoch = 1
    history = {key: [] for key in [
        'train_loss', 'overall_mae', 'male_mae', 'female_mae',
        'gender_gap', 'race_gap', 'epsilon', 'lambda_t', 'target_var'
    ]}
    for r in RACE_NAMES:
        history[f'{r}_mae'] = []

    if resume_from_epoch > 0:
        ckpt = CKPT.load_training_checkpoint(phase, epoch=resume_from_epoch)
        if ckpt is not None:
            print(f"  📂 Loading checkpoint for {phase.upper()} from epoch {resume_from_epoch}")
            base_model = model._module if hasattr(model, '_module') else model
            base_model.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            start_epoch = ckpt['epoch'] + 1
            history = ckpt['history']
            if 'dro_state' in ckpt and ckpt['dro_state']:
                q_t = ckpt['dro_state'].get('q_t', q_t)
                lambda_t = ckpt['dro_state'].get('lambda_t', 0.0)
            print(f"  ✅ Resumed from epoch {resume_from_epoch}. Starting from epoch {start_epoch}.")
        else:
            print(f"  ⚠️  Checkpoint for epoch {resume_from_epoch} not found. Starting from scratch.")
            start_epoch = 1

    # Wrap with Opacus
    privacy_engine = PrivacyEngine()
    model, optimizer, train_loader = privacy_engine.make_private(
        module=model, optimizer=optimizer, data_loader=train_loader,
        noise_multiplier=cfg['dp_noise_multiplier'], max_grad_norm=cfg['dp_max_grad_norm']
    )

    # Initial canary losses (only if starting fresh)
    if start_epoch == 1:
        with torch.no_grad():
            c_imgs, c_ages = canary_ds.get_all_canary_batch()
            initial_losses = (model(c_imgs.to(device)) - c_ages.to(device)).abs().cpu().numpy()
    else:
        # Load initial losses from checkpoint
        ckpt = CKPT.load_training_checkpoint(phase, epoch=resume_from_epoch)
        initial_losses = ckpt.get('extra', {}).get('initial_losses', None)
        if initial_losses is None:
            with torch.no_grad():
                c_imgs, c_ages = canary_ds.get_all_canary_batch()
                initial_losses = (model(c_imgs.to(device)) - c_ages.to(device)).abs().cpu().numpy()

    cumulative_delta_w = None
    wb_scores = np.zeros(k, dtype=np.float64)

    print(f'[{experiment_name}] Training epochs {start_epoch} to {cfg["epochs"]}...')
    start_time = time.time()

    try:
        for epoch in range(start_epoch, cfg['epochs'] + 1):
            # DRO sub-epoch sampling
            if is_alg1 and epoch > cfg['dro_warmup_epochs']:
                n_samples = int(n_train * 0.8)
                sampled_indices = np.random.choice(n_train, size=n_samples, p=q_t, replace=True)
                sampled_indices = list(dict.fromkeys(sampled_indices))
                temp_ds = TransformSubset(full_ds, [train_idx[i] for i in sampled_indices], train_tf)
                epoch_augmented_ds = ConcatDataset([temp_ds, in_canary_ds])
                train_loader = get_uniform_loader(epoch_augmented_ds)
            else:
                train_loader = get_uniform_loader(augmented_ds)

            model.train()
            epoch_loss = 0.0
            epoch_n = 0
            do_wb = (epoch % cfg['wb_score_epochs'] == 0 or epoch == cfg['epochs'])

            if do_wb:
                total_batches = len(train_loader)
                wb_batches = set(int(x) for x in np.random.choice(
                    total_batches, min(cfg['wb_sample_batches'], total_batches), replace=False
                ))
                epoch_delta_w = None

            with BatchMemoryManager(
                data_loader=train_loader,
                max_physical_batch_size=cfg['dp_phys_batch'],
                optimizer=optimizer
            ) as loader_iter:
                for batch_idx, (imgs, ages, _, _) in enumerate(loader_iter):
                    imgs, ages = imgs.to(device), ages.to(device)
                    params_before = None
                    if do_wb and int(batch_idx) in wb_batches:
                        params_before = [p.detach().clone() for p in model.parameters() if p.requires_grad]

                    loss = criterion(model(imgs), ages)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    epoch_loss += loss.item() * imgs.size(0)
                    epoch_n += imgs.size(0)

                    if params_before is not None:
                        batch_delta = torch.cat([
                            (b - a).view(-1)
                            for b, a in zip(params_before, [p.detach() for p in model.parameters() if p.requires_grad])
                        ])
                        epoch_delta_w = batch_delta if epoch_delta_w is None else epoch_delta_w + batch_delta

            # DRO update
            if is_alg1 and epoch >= cfg['dro_warmup_epochs']:
                model.eval()
                eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False, num_workers=2)
                train_losses = []
                with torch.no_grad():
                    for im, ag, _, _ in eval_loader:
                        train_losses.extend((model(im.to(device)) - ag.to(device)).abs().cpu().numpy())
                train_losses = np.array(train_losses)

                mu_t = np.sum(q_t * train_losses)
                V_t = np.sum(q_t * (train_losses - mu_t) ** 2)
                lambda_t = np.clip(lambda_t + eta * (alpha - V_t), 0.0, 1.0 / (2.0 * alpha))
                q_new = q_t * (1.0 + lambda_t * (train_losses - mu_t))
                q_new = np.maximum(q_new, 1e-12)
                q_t = q_new / q_new.sum()

            # White-box scoring
            if do_wb and epoch_delta_w is not None:
                cumulative_delta_w = epoch_delta_w if cumulative_delta_w is None else cumulative_delta_w + epoch_delta_w
                wb_model = build_model()
                wb_model.load_state_dict(model._module.state_dict())
                wb_model.eval()
                c_imgs, c_ages = canary_ds.get_all_canary_batch()
                wb_cont = np.zeros(k, dtype=np.float64)
                for s in range(0, k, 64):
                    e = min(s + 64, k)
                    sub_imgs = c_imgs[s:e].clone().detach().requires_grad_(True)
                    sub_ages = c_ages[s:e]
                    sub_grads = torch.autograd.grad(
                        (wb_model(sub_imgs) - sub_ages).abs().sum(),
                        list(wb_model.parameters())
                    )
                    grad_flat = torch.cat([g.view(-1) for g in sub_grads])
                    clip_f = min(1.0, cfg['dp_max_grad_norm'] / (grad_flat.norm().item() + 1e-8))
                    wb_cont[s:e] = torch.dot(cumulative_delta_w, grad_flat * clip_f).item() / len(wb_batches)
                wb_scores = wb_cont
                del wb_model
                torch.cuda.empty_cache()

            # Evaluation
            fair = evaluate_fairness(model, val_loader)
            for key in ['overall_mae', 'male_mae', 'female_mae', 'gender_gap', 'race_gap']:
                history[key].append(fair[key])
            for r in RACE_NAMES:
                history[f'{r}_mae'].append(fair[f'{r}_mae'])
            history['train_loss'].append(epoch_loss / epoch_n)
            history['epsilon'].append(privacy_engine.get_epsilon(cfg['dp_delta']))
            history['lambda_t'].append(lambda_t)
            history['target_var'].append(alpha)

            if epoch % 5 == 0:
                elapsed = time.time() - start_time
                print(f'  [{experiment_name}] Ep {epoch:2d} | L={epoch_loss/epoch_n:.3f} | '
                      f'Gap={fair["gender_gap"]:.2f} | ε={history["epsilon"][-1]:.2f} | {elapsed:.0f}s')

            # ── SAVE CHECKPOINT every N epochs ──
            if epoch % checkpoint_every == 0 or epoch == cfg['epochs']:
                dro_state = {'q_t': q_t, 'lambda_t': lambda_t} if is_alg1 else {}
                CKPT.save_training_checkpoint(
                    phase=phase,
                    epoch=epoch,
                    model=model,
                    optimizer=optimizer,
                    history=history,
                    dro_state=dro_state,
                    privacy_engine_steps=0,  # Opacus API changed; 0 is fine for resuming weights
                    extra={'initial_losses': initial_losses, 'wb_scores': wb_scores}
                )
                print(f"  💾 Checkpoint saved: {phase}/epoch_{epoch:03d}")

    except KeyboardInterrupt:
        # ── EMERGENCY SAVE on interruption ──
        current_epoch = epoch if 'epoch' in dir() else start_epoch
        print(f"\n  ⚠️  TRAINING INTERRUPTED at epoch {current_epoch}!")
        print(f"  💾 Saving emergency checkpoint...")
        dro_state = {'q_t': q_t, 'lambda_t': lambda_t} if is_alg1 else {}
        CKPT.save_training_checkpoint(
            phase=phase,
            epoch=current_epoch,
            model=model,
            optimizer=optimizer,
            history=history,
            dro_state=dro_state,
            privacy_engine_steps=0,  # Opacus API changed
            extra={'initial_losses': initial_losses, 'wb_scores': wb_scores}
        )
        # Update metadata
        metadata = CKPT.load_metadata() or {}
        metadata['current_stage'] = f'{phase}_training'
        metadata['last_completed_epoch'] = current_epoch
        metadata['status'] = 'interrupted'
        CKPT.save_metadata(metadata)
        print(f"  ✅ Emergency checkpoint saved. You can safely resume later.")
        return None  # Signal that training was interrupted

    # Final canary losses
    c_imgs, c_ages = canary_ds.get_all_canary_batch()
    with torch.no_grad():
        final_losses = (model(c_imgs.to(device)) - c_ages.to(device)).abs().cpu().numpy()

    print(f'  [{experiment_name}] Done in {time.time()-start_time:.0f}s')

    return {
        'model': model,
        'bb_scores': initial_losses - final_losses,
        'wb_scores': wb_scores,
        'history': history,
        'epsilon_spent': history['epsilon'][-1],
        'final_metrics': fair
    }

print("✅ Smart training function defined.")

✅ Smart training function defined.


## § 7. Main Execution: Smart Decision Engine

In [8]:
# ══════════════════════════════════════════════════════════════════
# MAIN EXECUTION: SMART DECISION ENGINE
# ══════════════════════════════════════════════════════════════════

# Build a temporary model to compute model hash for change detection
temp_model = build_model()

# ── MAKE THE DECISION ──
decision, reason, context = CKPT.decide_action(CFG, temp_model)
CKPT.print_decision(decision, reason)

# ── EXECUTE BASED ON DECISION ──
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=4, pin_memory=True)

if decision == RunDecision.COMPLETE:
    print("  Loading previously saved results...")
    final_results = CKPT.load_final_results()
    print("  ✅ Results loaded. Skip to the plotting cell.")

elif decision == RunDecision.START_FRESH:
    # Archive previous run if it exists
    archive_path = CKPT.archive_previous_run()
    if archive_path:
        print(f"  📦 Previous run archived to: {archive_path}")

    # Save hashes for future comparison
    CKPT.save_hashes(CFG, temp_model)
    del temp_model
    gc.collect()
    torch.cuda.empty_cache()

    # Initialize metadata
    CKPT.save_metadata({
        'status': 'running',
        'current_stage': 'erm_training',
        'completed_phases': [],
        'start_time': datetime.now().isoformat(),
        'config': {k: str(v) for k, v in CFG.items()}
    })

    # ── TRAIN ERM ──
    print('\n' + '═' * 78)
    print('  EXPERIMENT 1: DP-SGD + ERM (Baseline)')
    print('═' * 78)
    cfg_erm = CFG.copy()
    cfg_erm['dro_mode'] = 'erm'
    res_erm = train_model_with_checkpoints(
        cfg_erm, train_ds, val_loader, canary_ds, membership,
        phase='erm', experiment_name='ERM', resume_from_epoch=0
    )

    if res_erm is None:
        print("\n  ⛔ ERM training was interrupted. Run this cell again to resume.")
    else:
        # Quality check before proceeding
        is_good, quality_msg = CKPT.assess_training_quality(res_erm['history'], 'erm')
        if not is_good:
            print(f"\n  ❌ ERM results failed quality check: {quality_msg}")
            print("  🔄 Starting fresh on next run.")
        else:
            print(f"\n  ✅ ERM quality check passed: {quality_msg}")
            # Update metadata
            metadata = CKPT.load_metadata()
            metadata['completed_phases'].append('erm_training')
            metadata['current_stage'] = 'alg1_training'
            CKPT.save_metadata(metadata)

            # ── TRAIN ALG1 ──
            print('\n' + '═' * 78)
            print('  EXPERIMENT 2: DP-SGD + Hashimoto ALG1 (Exact Paper Implementation)')
            print('═' * 78)
            cfg_alg1 = CFG.copy()
            cfg_alg1['dro_mode'] = 'alg1_chisq'
            res_alg1 = train_model_with_checkpoints(
                cfg_alg1, train_ds, val_loader, canary_ds, membership,
                phase='alg1', experiment_name='ALG1', resume_from_epoch=0
            )

            if res_alg1 is None:
                print("\n  ⛔ ALG1 training was interrupted. Run this cell again to resume.")
            else:
                # Quality check
                is_good, quality_msg = CKPT.assess_training_quality(res_alg1['history'], 'alg1')
                if not is_good:
                    print(f"\n  ❌ ALG1 results failed quality check: {quality_msg}")
                else:
                    print(f"\n  ✅ ALG1 quality check passed: {quality_msg}")
                    metadata = CKPT.load_metadata()
                    metadata['completed_phases'].append('alg1_training')
                    metadata['current_stage'] = 'audit'
                    CKPT.save_metadata(metadata)
                    print("\n  ✅ Both training phases complete. Proceed to audit cell.")

elif decision == RunDecision.RESUME_TRAINING:
    # Determine where to resume
    metadata = CKPT.load_metadata()
    completed = metadata.get('completed_phases', [])

    if 'erm_training' in completed:
        # Resume ALG1 training
        print("  📂 Resuming ALG1 training from previous checkpoint...")
        alg1_ckpt = CKPT.load_training_checkpoint('alg1')
        resume_epoch = alg1_ckpt['epoch'] if alg1_ckpt else 0

        # Load ERM results
        erm_ckpt = CKPT.load_training_checkpoint('erm')
        res_erm = {
            'history': erm_ckpt['history'],
            'epsilon_spent': erm_ckpt['history']['epsilon'][-1],
            'final_metrics': {
                'overall_mae': erm_ckpt['history']['overall_mae'][-1],
                'gender_gap': erm_ckpt['history']['gender_gap'][-1],
                'race_gap': erm_ckpt['history']['race_gap'][-1],
            }
        }
        # We need bb/wb scores - load from checkpoint extra
        res_erm['bb_scores'] = erm_ckpt.get('extra', {}).get('bb_scores')
        res_erm['wb_scores'] = erm_ckpt.get('extra', {}).get('wb_scores')

        # Train ALG1 from checkpoint
        cfg_alg1 = CFG.copy()
        cfg_alg1['dro_mode'] = 'alg1_chisq'
        res_alg1 = train_model_with_checkpoints(
            cfg_alg1, train_ds, val_loader, canary_ds, membership,
            phase='alg1', experiment_name='ALG1', resume_from_epoch=resume_epoch
        )
        if res_alg1 is not None:
            metadata = CKPT.load_metadata()
            metadata['completed_phases'].append('alg1_training')
            metadata['current_stage'] = 'audit'
            CKPT.save_metadata(metadata)
    else:
        # Resume ERM training from partial
        erm_ckpt = CKPT.load_training_checkpoint('erm')
        resume_epoch = erm_ckpt['epoch'] if erm_ckpt else 0
        print(f"  📂 Resuming ERM training from epoch {resume_epoch}...")

        cfg_erm = CFG.copy()
        cfg_erm['dro_mode'] = 'erm'
        res_erm = train_model_with_checkpoints(
            cfg_erm, train_ds, val_loader, canary_ds, membership,
            phase='erm', experiment_name='ERM', resume_from_epoch=resume_epoch
        )
        if res_erm is not None:
            metadata = CKPT.load_metadata()
            metadata['completed_phases'].append('erm_training')
            metadata['current_stage'] = 'alg1_training'
            CKPT.save_metadata(metadata)
            # Then proceed to ALG1 (run this cell again)

elif decision == RunDecision.RESUME_AUDIT:
    print("  📂 Loading trained models from checkpoints for auditing...")
    erm_ckpt = CKPT.load_training_checkpoint('erm')
    alg1_ckpt = CKPT.load_training_checkpoint('alg1')
    print("  ✅ Models loaded. Proceed to audit cell.")

elif decision == RunDecision.RESUME_PLOTTING:
    print("  📂 Loading audit results from checkpoint for plotting...")
    audit_results = CKPT.load_audit_results()
    print("  ✅ Audit results loaded. Proceed to plotting cell.")

print("\n✅ Decision engine complete.")


══════════════════════════════════════════════════════════════════════════════
  🔄 DECISION: STARTING FRESH RUN
══════════════════════════════════════════════════════════════════════════════
  Reason: STARTING FRESH: Could not determine a valid resume point. Starting over.
══════════════════════════════════════════════════════════════════════════════

  📦 Previous run archived to: project/experiments/paper_accurate_final/checkpoints_archived_20260923_232900

══════════════════════════════════════════════════════════════════════════════
  EXPERIMENT 1: DP-SGD + ERM (Baseline)
══════════════════════════════════════════════════════════════════════════════
[ERM] Training epochs 1 to 25...
  [ERM] Ep  5 | L=16.220 | Gap=2.29 | ε=0.46 | 211s
  💾 Checkpoint saved: erm/epoch_005
  [ERM] Ep 10 | L=15.158 | Gap=1.62 | ε=0.65 | 431s
  💾 Checkpoint saved: erm/epoch_010
  [ERM] Ep 15 | L=13.856 | Gap=0.51 | ε=0.81 | 653s
  💾 Checkpoint saved: erm/epoch_015
  [ERM] Ep 20 | L=13.043 | Gap=0.12 | ε=0

---
## § 8. Privacy Audit Phase (with checkpoint)

In [9]:
# ══════════════════════════════════════════════════════════════════
# PRIVACY AUDIT PHASE
# ══════════════════════════════════════════════════════════════════

# Check if audit was already done
audit_results = CKPT.load_audit_results()

if audit_results is not None:
    print("  📂 Audit results found in checkpoint. Skipping audit.")
    bb_erm = audit_results['bb_erm']
    wb_erm = audit_results['wb_erm']
    bb_alg1 = audit_results['bb_alg1']
    wb_alg1 = audit_results['wb_alg1']
else:
    print("  🔍 Running privacy audit...")

    # Run audits
    bb_erm = audit_epsilon(res_erm['bb_scores'], membership)
    wb_erm = audit_epsilon(res_erm['wb_scores'], membership)
    bb_alg1 = audit_epsilon(res_alg1['bb_scores'], membership)
    wb_alg1 = audit_epsilon(res_alg1['wb_scores'], membership)

    # Save audit results
    audit_results = {
        'bb_erm': bb_erm,
        'wb_erm': wb_erm,
        'bb_alg1': bb_alg1,
        'wb_alg1': wb_alg1,
        'timestamp': datetime.now().isoformat()
    }
    CKPT.save_audit_results(audit_results)
    print("  💾 Audit results saved to checkpoint.")

    # Update metadata
    metadata = CKPT.load_metadata()
    metadata['completed_phases'].append('audit')
    metadata['current_stage'] = 'plotting'
    CKPT.save_metadata(metadata)

print(f"\n  ERM  → BB ε_lb: {bb_erm['eps_lb']:.4f}, WB ε_lb: {wb_erm['eps_lb']:.4f}")
print(f"  ALG1 → BB ε_lb: {bb_alg1['eps_lb']:.4f}, WB ε_lb: {wb_alg1['eps_lb']:.4f}")
print("✅ Audit complete.")

  🔍 Running privacy audit...
  💾 Audit results saved to checkpoint.

  ERM  → BB ε_lb: 0.0000, WB ε_lb: 0.0000
  ALG1 → BB ε_lb: 0.0000, WB ε_lb: 0.0000
✅ Audit complete.


## § 9. Save Final Results & Mark Experiment Complete

In [10]:
# ══════════════════════════════════════════════════════════════════
# SAVE FINAL RESULTS
# ══════════════════════════════════════════════════════════════════

def convert_to_serializable(obj):
    if isinstance(obj, np.ndarray): return obj.tolist()
    elif isinstance(obj, (np.integer,)): return int(obj)
    elif isinstance(obj, (np.floating,)): return float(obj)
    elif isinstance(obj, np.bool_): return bool(obj)
    elif isinstance(obj, Path): return str(obj)
    elif isinstance(obj, dict): return {convert_to_serializable(k): convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)): return [convert_to_serializable(x) for x in obj]
    elif isinstance(obj, torch.Tensor): return obj.tolist()
    else: return obj

final_results = {
    'timestamp': datetime.now().isoformat(),
    'experiment_name': 'DP-SGD + Hashimoto ALG1 + Smart Checkpointing',
    'config': convert_to_serializable(CFG),
    'erm': {
        'history': convert_to_serializable(res_erm['history']),
        'epsilon_spent': float(res_erm['epsilon_spent']),
        'bb_audit': convert_to_serializable(bb_erm),
        'wb_audit': convert_to_serializable(wb_erm),
        'final_metrics': convert_to_serializable(res_erm.get('final_metrics', {})),
    },
    'alg1': {
        'history': convert_to_serializable(res_alg1['history']),
        'epsilon_spent': float(res_alg1['epsilon_spent']),
        'bb_audit': convert_to_serializable(bb_alg1),
        'wb_audit': convert_to_serializable(wb_alg1),
        'final_metrics': convert_to_serializable(res_alg1.get('final_metrics', {})),
        'final_lambda_t': float(res_alg1['history']['lambda_t'][-1]),
    },
    'comparison': {
        'gender_gap_reduction_pct': float(
            (res_erm['history']['gender_gap'][-1] - res_alg1['history']['gender_gap'][-1])
            / max(res_erm['history']['gender_gap'][-1], 0.01) * 100
        ),
        'race_gap_reduction_pct': float(
            (res_erm['history']['race_gap'][-1] - res_alg1['history']['race_gap'][-1])
            / max(res_erm['history']['race_gap'][-1], 0.01) * 100
        ),
        'utility_cost': float(res_alg1['history']['overall_mae'][-1] - res_erm['history']['overall_mae'][-1]),
    }
}

CKPT.save_final_results(final_results)

# Mark experiment as complete
metadata = CKPT.load_metadata()
metadata['status'] = 'complete'
metadata['completed_phases'].append('plotting')
metadata['end_time'] = datetime.now().isoformat()
CKPT.save_metadata(metadata)

print("✅ Final results saved.")
print("✅ Experiment marked as COMPLETE.")
print(f"   Results: {CKPT.ckpt_dir / 'final_results.json'}")
print(f"   Decision log: {CKPT.decision_log_file}")

✅ Final results saved.
✅ Experiment marked as COMPLETE.
   Results: project/experiments/paper_accurate_final/checkpoints/final_results.json
   Decision log: project/experiments/paper_accurate_final/checkpoints/decision_log.json
